In [126]:
import weaviate
from weaviate.classes.query import (Filter, Rerank)



### Setting up the weaviate client and loading the data

In [127]:
client = weaviate.connect_to_local(port= 8080)

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\weaviate\warnings.py:93: DeprecationWarning: Dep005: You are using weaviate-client version 4.16.2. The latest version is 4.22.0.
            Consider upgrading to the latest version. See https://weaviate.io/developers/weaviate/client-libraries/python for details.
  warnings.warn(


In [128]:
# loading the data set

import joblib
bbc_data = joblib.load('data/bbc_data.joblib')
print("les données ont été chargé avec succés!")

c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\site-packages\weaviate\warnings.py:292: ResourceWarning: Con004: The connection to Weaviate was not closed properly. This can lead to memory leaks.
            Please make sure to close the connection using `client.close()`.
  warnings.warn(
c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\pickle.py:1590: ResourceWarning: unclosed <socket.socket fd=6656, family=AddressFamily.AF_INET6, type=SocketKind.SOCK_STREAM, proto=0, laddr=('::1', 57501, 0, 0), raddr=('::1', 8080, 0, 0)>
  stack[-1] = func(*args)
c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\pickle.py:1590: ResourceWarning: unclosed <socket.socket fd=6296, family=AddressFamily.AF_INET6, type=SocketKind.SOCK_STREAM, proto=0, laddr=('::1', 57373, 0, 0), raddr=('::1', 8080, 0, 0)>
  stack[-1] = func(*args)
c:\Users\hp\AppData\Local\Programs\Python\Python310\lib\pickle.py:1590: ResourceWarning: unclosed <socket.socket fd=6412, family=AddressFamily.AF_INET6, type=

les données ont été chargé avec succés!


In [129]:
len(bbc_data)

9973

In [130]:
bbc_data[0]

{'title': 'Justin Welby: Political leaders should treat opponents as human beings',
 'pubDate': Timestamp('2024-01-01 00:00:04'),
 'guid': 'https://www.bbc.co.uk/news/uk-67844356',
 'link': 'https://www.bbc.co.uk/news/uk-67844356?at_medium=RSS&at_campaign=KARANGA',
 'description': 'The Archbishop of Canterbury urges politicians to "forswear wedge issues" and avoid divisive topics.',
 'article_content': 'Justin Welby speaks on BBC Radio 4\'s Today programme as part of a special show guest edited by Dame Emma Warmsley The Archbishop of Canterbury has urged politicians not to treat their opponents as enemies but fellow human beings. Speaking to the BBC, the Most Rev Justin Welby warned Britain\'s leaders to avoid divisive topics. But he said our capacity "to disagree deeply and not destructively" is cause for hope. Later, he will deliver a new year\'s message reflecting on global conflicts and his wishes for a "peaceful 2024". The archbishop\'s intervention came during an interview for BB

In [131]:
bbc_data[0]['title']

'Justin Welby: Political leaders should treat opponents as human beings'

On a 6 colonnnes:

- title
- pubDate
- guid
- link
- description
- article-content

In [132]:
# Loading the collection
collection = client.collections.get("bbc_collection")


In [ ]:
# Insérer les éléments dans la collection
from weaviate.classes. config import Configure


if not client.collections.exists("bbc_collection"):
    
    # 1- créer la collection
    print("la collection n'esxiste pas. Création en cours ...")
    collection = client.collections.create(
        name= "bbc_collection",
    )
    
    # 2- insérer les données dans la collection 
    print("Insertion des données de bbc_data.joblib dans la collection ...")
    with collection.batch.dynamic() as batch:
        for item in bbc_data:
            batch.add_object(properties=
                            {
            "title": item.get("title", ""),
            "pubDate": item.get("pubDate", ""),
            "guid": item.get("guid", ""),
            "link": item.get("link", ""),
            "description": item.get("description", ""),
            "article_content": item.get("article-content", "")  
            })
            
else:
    collection = client.collections.get("bbc_collection")
    print("la collection existe déjà!")
    


la collection existe déjà!


In [134]:
print(f"Le nombre d'éléments dans la collection est {len(collection)}")

Le nombre d'éléments dans la collection est 9973


## 1. Metadata filtering

In [135]:
# we will implement a meta data filtering 
# this function retrieves objects from a specified collection based on metadata filtering criteria
def filter_by_metadata( metadata_property: str,
                        values: list[str],
                        collection: "weaviate.collections.collection.sync.Collection",
                        limit: int= 5 ) -> list:
    
    
    response = collection.query.fetch_objects(
        filters = Filter.by_property(metadata_property).contains_any(values),
        limit= limit)
    
    response_objects = [x.properties for x in response.objects]
    return response_objects    
    
    


    

In [136]:
# Par exemple 
# metadata_property = 'title
# values = ['Taylor Swift']
# limit = 2


res = filter_by_metadata('title', ['Taylor Swift'], collection, limit = 2)
for x in res:
    print(x['title'])

Margot Robbie, Taylor Swift and more on Golden Globes red carpet
TikTok mutes users' videos as it pulls Taylor Swift and The Weeknd's music


## 2. Semantic search

In [137]:
def semantic_search_retrieve(
                                query: str,
                                collection: "weaviate.collections.collection.sync.Collection",
                                top_k: int = 5) -> list:
    
    response = collection.query.near_text(
        query = query,
        limit= top_k
    )
    
    response_objects = [x.properties for x in response.objects]
    return response_objects

In [140]:
"""semantic_search_retrieve(
    query = 'Tell me about the last Taylor Swift show', 
    collection = collection, 
    top_k = 2)"""
print()

# erreur car weavaite doit transformer la requte en un vecteur, et cela necessite une api comme de text2vec de openai payant!!!


<b>Normalement le résultat doit etre :</b>
<pre>
article_content: Taylor Swift has finished the European leg of her Eras Tour with a record-breaking show at Wembley S...(truncated)
chunk: size crowd at all". At an earlier show in Liverpool, she had also called the Eras Tour the “most exh...(truncated)
chunk_index: 10
description: The star is joined by Florence + The Machine and sings So Long, London at her final UK show.
link: https://www.bbc.com/news/articles/cr5nr3n6epvo
pubDate: 2024-08-21 03:02:08+00:00
title: 'I've never had it this good' - Taylor Swift thanks fans after new Wembley record

article_content: Taylor Swift has finished the European leg of her Eras Tour with a record-breaking show at Wembley S...(truncated)
chunk: regular part of the setlist. Last week, the star was joined by Ed Sheeran to play the songs Endgame ...(truncated)
chunk_index: 4
description: The star is joined by Florence + The Machine and sings So Long, London at her final UK show.
link: https://www.bbc.com/news/articles/cr5nr3n6epvo
pubDate: 2024-08-21 03:02:08+00:00
title: 'I've never had it this good' - Taylor Swift thanks fans after new Wembley record

## 3- BM25 Search

In [141]:
def bm25_retrieve(query: str, 
                  collection: "weaviate.collections.collection.sync.Collection" , 
                  top_k: int = 5) -> list:
    
    


    # Retrieve using collection.query.bm25
    response = collection.query.bm25(
        query= query,
        limit= top_k
    )

  
    
    response_objects = [x.properties for x in response.objects]
    return response_objects 

In [142]:
bm25_retrieve('Tell me about the last Taylor Swift show', collection, top_k = 2)

[{'guid': 'https://www.bbc.co.uk/news/uk-england-london-68876725#0',
  'title': "Taylor Swift fans 'overwhelming' London pub",
  'pubDate': datetime.datetime(2024, 4, 22, 16, 32, 51, tzinfo=datetime.timezone.utc),
  'link': 'https://www.bbc.co.uk/news/uk-england-london-68876725',
  'description': 'A south London pub named on Taylor Swift\'s new album describes the reaction as "crazy".',
  'article_content': ''},
 {'guid': 'https://www.bbc.co.uk/news/world-australia-68271324',
  'title': "Taylor Swift: Inside a world-first 'Swiftposium' academic summit",
  'pubDate': datetime.datetime(2024, 2, 12, 16, 3, 50, tzinfo=datetime.timezone.utc),
  'link': 'https://www.bbc.co.uk/news/world-australia-68271324?at_medium=RSS&at_campaign=KARANGA',
  'description': "The BBC goes inside the world's most dedicated effort to chart Taylor Swift's power and influence.",
  'article_content': ''}]

## 4. Hybrid search

In [143]:
def hybrid_retrieve(query: str, 
                    collection: "weaviate.collections.collection.sync.Collection" , 
                    alpha: float = 0.5,
                    top_k: int = 5
                   ) -> list:



    # Retrieve using collection.query.hybrid
    response = collection.query.hybrid(
        query= query,
        alpha= alpha,
        limit= top_k
    )


    
    response_objects = [x.properties for x in response.objects]
    
    return response_objects 

In [ ]:
"""hybrid_retrieve('Tell me about the last Taylor Swift show', collection, top_k = 2)"""

WeaviateQueryError: Query call with protocol GRPC search failed with message get vector input from modules provider: VectorFromInput was called without vectorizer.

<b> De meme nécessite une API payante, le résultat est </b>

<pre>
article_content: Rapper Killer Mike won three Grammys in the rap category - best rap song, best rap performance and b...(truncated)
chunk: police brutality and systemic racism. He was a highly visible supporter of Bernie Sanders' two campa...(truncated)
chunk_index: 4
description: The 48-year-old was detained on a misdemeanour charge after winning three awards in the rap category.
link: https://www.bbc.co.uk/news/world-us-canada-68201021?at_medium=RSS&at_campaign=KARANGA
pubDate: 2024-02-05 23:27:08+00:00
title: Killer Mike dismisses arrest at Grammys as 'speed bump'

article_content: Taylor Swift has finished the European leg of her Eras Tour with a record-breaking show at Wembley S...(truncated)
chunk: size crowd at all". At an earlier show in Liverpool, she had also called the Eras Tour the “most exh...(truncated)
chunk_index: 10
description: The star is joined by Florence + The Machine and sings So Long, London at her final UK show.
link: https://www.bbc.com/news/articles/cr5nr3n6epvo
pubDate: 2024-08-21 03:02:08+00:00
title: 'I've never had it this good' - Taylor Swift thanks fans after new Wembley record

## 5- Reranking


In [ ]:
# GRADED CELL 

def semantic_search_with_reranking(query: str, 
                                   rerank_property: str,
                                   collection: "weaviate.collections.collection.sync.Collection" , 
                                   rerank_query: str = None,
                                   top_k: int = 5
                                   ) -> list:


    if rerank_query is None: 
        rerank_query = query 
        
    # Define the reranker with rerank_query and rerank_property
    reranker = Rerank(query= rerank_query, prop= rerank_property)

    response = collection.query.near_text(
        query = query,
        rerank = reranker,
        limit= top_k
    )


    
    response_objects = [x.properties for x in response.objects]
    
    return response_objects 

In [ ]:
query = 'Tell me about the conflicts in Latin America'
results = semantic_search_with_reranking(query, collection = collection, top_k = 2, rerank_property = 'chunk')